In [2]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import sys

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

import time
import gc

DATASET_PATH = Path("../data/text")

VOC_SIZE = 1000

def load_data(datapath, max_size=None):
    texts_files = list(datapath.glob("*.txt"))
    texts = []  
    for files in texts_files:
        with open(files, "r", encoding='utf8') as files:
            text = files.readlines()
            texts += text
    texts = list(set(texts))
    
    return texts

texts = load_data(DATASET_PATH)

from tokenizers import Tokenizer
# from transformers import AutoTokenizer
from transformers import BertForMaskedLM, AutoTokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace


model_checkpoint = "bert-base-uncased"

tokenizerBERT_FT = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)
modelBERT_FT = BertForMaskedLM.from_pretrained(model_checkpoint)

inputs = tokenizerBERT_FT(texts, return_tensors='pt', max_length=100
                   , truncation=True, padding='max_length')

inputs['labels'] = inputs.input_ids.detach().clone()

print(inputs.tokens(1))


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


['[CLS]', 'au', '##tres', 'pro', '##du', '##its', 've', '##get', '##aux', 'go', '##uss', '##es', 'vi', '##des', 'de', 'ba', '##uh', '##inia', '(', 'ba', '##uh', '##inia', 'tho', '##nni', '##ng', '##ii', ')', 'go', '##uss', '##es', 'vi', '##des', 'de', 'ba', '##uh', '##inia', '(', 'ba', '##uh', '##inia', 'tho', '##nni', '##ng', '##ii', 'sc', '##hum', '.', ')', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


In [8]:
CLS_id = tokenizerBERT_FT.encode('[CLS]')[1]
EOS_id = tokenizerBERT_FT.encode('[EOS]')[1]
PAD_id = tokenizerBERT_FT.encode('[PAD]')[1]
UNK_id = tokenizerBERT_FT.encode('[UNK]')[1]
SEP_id = tokenizerBERT_FT.encode('[SEP]')[1]

print(f'PAD token id : {PAD_id}')
print(f'EOS token id : {EOS_id}')
print(f'CLS token id : {CLS_id}')
print(f'UNK token id : {UNK_id}')
print(f'SEP token id : {SEP_id}')

PAD token id : 0
EOS token id : 1031
CLS token id : 101
UNK token id : 100
SEP token id : 102


In [6]:
tokenizerBERT_FT.special_tokens_map

{'unk_token': '[UNK]',
 'sep_token': '[SEP]',
 'pad_token': '[PAD]',
 'cls_token': '[CLS]',
 'mask_token': '[MASK]'}

In [9]:
rand = torch.rand(inputs.input_ids.shape)
mask_arr = (rand < 0.15) * (inputs.input_ids != SEP_id) * (inputs.input_ids != CLS_id) * (inputs.input_ids != SEP_id)

In [10]:
import random as rd

In [11]:
# apply the [MASK] token with the mask array 
inputs.input_ids[mask_arr] = 103

class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, idx):
        self.encodings = encodings
        self.idx = idx
        self.encodings = {key: [val[i] for i in self.idx] for key, val in self.encodings.items()}
        
    def __getitem__(self, idx):
        return {key : torch.tensor(val[idx]) for key, val in self.encodings.items()}
    
    def __len__(self):
        return len(self.encodings['input_ids'])

sample_idx = [i for i in range(len(inputs.input_ids))]

shuffled_sample_idx = rd.sample(sample_idx, len(sample_idx))

train_idx = shuffled_sample_idx[:int(0.70*len(shuffled_sample_idx))]
val_idx = shuffled_sample_idx[int(0.70*len(shuffled_sample_idx)):int(0.85*len(shuffled_sample_idx))]
test_idx = shuffled_sample_idx[int(0.85*len(shuffled_sample_idx)):]
                                
dataset_train = CustomDataset(inputs, train_idx)
dataset_val = CustomDataset(inputs, val_idx)
dataset_test = CustomDataset(inputs, test_idx)

train_dataloaded = torch.utils.data.DataLoader(dataset_train, batch_size=16, shuffle=True)
val_dataloaded = torch.utils.data.DataLoader(dataset_val, batch_size=16, shuffle=True)
test_dataloaded = torch.utils.data.DataLoader(dataset_test, batch_size=16, shuffle=True)

In [12]:
#class MLM_model(nn.Module):
#    def __init__(self, model):
#        super(MLM_model, self).__init__()
#        self.history = {"epochs":[], "test":[]}
#        self.model = model
    
#    def parameters(self):
#        return self.model.parameters()

#    def forward(self, x, attention_mask, labels):
#        return self.model(x, attention_mask, labels)
    
#    def train_log(self, train_batch_losses, val_batch_losses, train_loss, validation_loss):
#        self.history["epochs"].append({"train_batch_losses":train_batch_losses, 
#                                "val_batch_losses":val_batch_losses, 
#                                "train_loss":train_loss, 
#                                "validation_loss":validation_loss})
    
#    def test_log(self, test_batch_losses, test_loss):
#        self.history["test"].append({"test_batch_losses":test_batch_losses,
#                                "test_loss":test_loss})

In [13]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
#model = MLM_model(model)
modelBERT_FT.to(device)
print(device)

cpu


In [14]:
def train_step(module, batch, batch_idx, optimizer):
    module.train(True)
    
    inputs_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)
    
    outputs = module(inputs_ids, attention_mask, labels=labels)
    
    loss = outputs.loss
    print(f"\n\033[1;37mBatch loss {batch_idx+1} : {loss.item()}")
    loss.backward()
    
    torch.nn.utils.clip_grad_norm_(module.parameters(), max_norm=1.0)
    optimizer.step()
    optimizer.zero_grad()
    
    return module, loss

def eval_step(module, batch, batch_idx, optimizer=None, training=True):
    if training == False :
        module.to('cpu')
    with torch.no_grad():
        
        inputs_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
    
        outputs = module(inputs_ids, attention_mask, labels=labels)
    
        loss = outputs.loss
         
        if training:
            print(f"\n\033[1;32mValidation Batch loss {batch_idx+1} : {loss.item()}")
            return module, loss
        else:
            print(f"\n\033[1;32mTest Batch loss {batch_idx+1} : {loss.item()}")
            return module, loss, outputs, labels

def train_loop(module, EPOCHS, train_dataset, val_dataset, optimizer, lr_scheduler=None):
    for epoch in range(EPOCHS):
        deb=time.time()
        
        module.train(True)
        
        train_batch_losses = []
        for batch_idx in range(len(train_dataset)):
            batch = next(iter(train_dataset))
            module, loss = train_step(module, batch, batch_idx, optimizer)
            train_batch_losses.append(loss.item())
            
        if lr_scheduler is not None:
          lr_scheduler.step()
        train_loss = np.mean(train_batch_losses)

        module.train(False)
        val_batch_losses = []
        for batch_idx in range(len(val_dataset)):
            batch = next(iter(val_dataset))
            module, loss = eval_step(module, batch, batch_idx)
            val_batch_losses.append(loss.item())
        val_loss = np.mean(val_batch_losses)

#        module.train_log(train_batch_losses, val_batch_losses, train_loss, val_loss)
        print(f"\n\033[1;33mEpoch {epoch+1} :\n\033[1;37mTraining Loss : {train_loss}")
        print(f"\033[1;32mValidation Loss : {val_loss}")
        print(f"\033[1;31mDurée epoch : {time.time()-deb} secondes")
    return module

def evaluate(module, test_dataset):
    module.train(False)
    test_batch_losses = []
    predictions = []
    true_targets = []
    for batch_idx in range(len(test_dataset)):
        batch = next(iter(test_dataset))
        module, loss, outputs, labels = eval_step(module, batch, batch_idx, training=False)

        test_batch_losses.append(loss.item())
        predictions.append(outputs)
        true_targets.append(labels)

    test_loss = np.mean(test_batch_losses)
#    module.test_log(test_batch_losses, test_loss)
    print(f"\nTest Loss : {test_loss}")
    return predictions, true_targets

In [15]:
if __name__ == "__main__":
    EPOCHS = 1
    LR = 1e-4
    
    optimizer = torch.optim.Adam(modelBERT_FT.parameters(), lr=LR, eps=5e-8)
    module = train_loop(module=modelBERT_FT,
                        EPOCHS=EPOCHS, 
                        train_dataset=train_dataloaded, 
                        val_dataset=val_dataloaded,
                        optimizer=optimizer)
    device = 'cpu'
    predictions, true_targets = evaluate(module, 
                                         test_dataloaded)



C:\Users\rapha\AppData\Local\Temp\ipykernel_32572\2355751477.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return {key : torch.tensor(val[idx]) for key, val in self.encodings.items()}



Batch loss 1 : 11.555730819702148

Batch loss 2 : 7.330380439758301


KeyboardInterrupt: 

In [9]:
print(torch.cuda.memory_stats())

OrderedDict({'active.all.allocated': 1087630, 'active.all.current': 406, 'active.all.freed': 1087224, 'active.all.peak': 1024, 'active.large_pool.allocated': 506367, 'active.large_pool.current': 152, 'active.large_pool.freed': 506215, 'active.large_pool.peak': 381, 'active.small_pool.allocated': 581263, 'active.small_pool.current': 254, 'active.small_pool.freed': 581009, 'active.small_pool.peak': 721, 'active_bytes.all.allocated': 4169757224960, 'active_bytes.all.current': 894567424, 'active_bytes.all.freed': 4168862657536, 'active_bytes.all.peak': 3127888384, 'active_bytes.large_pool.allocated': 4083723288064, 'active_bytes.large_pool.current': 893321216, 'active_bytes.large_pool.freed': 4082829966848, 'active_bytes.large_pool.peak': 3116457984, 'active_bytes.small_pool.allocated': 86033936896, 'active_bytes.small_pool.current': 1246208, 'active_bytes.small_pool.freed': 86032690688, 'active_bytes.small_pool.peak': 13091328, 'allocated_bytes.all.allocated': 4169757224960, 'allocated_by

In [10]:
print(inputs.input_ids.max())
print(inputs.input_ids.min())

tensor(29674)
tensor(0)


In [11]:
# enregistrement du modèle

modelBERT_FT.save_pretrained('./saves/model/BERT_FT')
tokenizerBERT_FT.save_pretrained('./saves/tokenizer/BERT_FT')

('./saves/tokenizer/BERT_FT\\tokenizer_config.json',
 './saves/tokenizer/BERT_FT\\special_tokens_map.json',
 './saves/tokenizer/BERT_FT\\vocab.txt',
 './saves/tokenizer/BERT_FT\\added_tokens.json',
 './saves/tokenizer/BERT_FT\\tokenizer.json')